# Analise de negocio

Este notebook responde as duas hipoteses de negocio definidas para este MVP, substituindo as
perguntas genericas da versao anterior (atraso->satisfacao simples; vendas por estado):

1. **Frete vs. porte fisico do produto:** quais categorias pagam mais frete por quilo/volume
   transportado, e isso e proporcional ao porte fisico ou indica ineficiencia logistica/pricing?
2. **Concentracao de problemas por vendedor:** um grupo pequeno de vendedores concentra a maior
   parte dos atrasos e notas baixas (padrao 80/20), ou o problema esta distribuido por todos?

Fontes: `${catalog}.olist_gold.fato_vendas` e `${catalog}.olist_gold.dim_produtos`. Cada consulta
e seguida de uma celula markdown de interpretacao, a ser preenchida com os valores reais obtidos
na execucao no Databricks.


In [0]:
-- Pergunta 1: custo de frete relativo ao porte fisico, por categoria de produto.
WITH item_metrics AS (
  SELECT
    f.order_id,
    f.product_id,
    f.freight_value,
    p.product_category_name,
    p.product_weight_g,
    (p.product_length_cm * p.product_height_cm * p.product_width_cm) / 1000000.0 AS volume_m3
  FROM ${catalog}.olist_gold.fato_vendas f
  JOIN ${catalog}.olist_gold.dim_produtos p USING (product_id)
  WHERE p.product_weight_g IS NOT NULL
    AND p.product_length_cm IS NOT NULL
    AND p.product_height_cm IS NOT NULL
    AND p.product_width_cm IS NOT NULL
),
categoria_agg AS (
  SELECT
    product_category_name,
    COUNT(*) AS total_itens,
    ROUND(AVG(freight_value), 2) AS frete_medio,
    ROUND(AVG(product_weight_g), 0) AS peso_medio_g,
    ROUND(AVG(volume_m3), 4) AS volume_medio_m3,
    ROUND(AVG(freight_value) / NULLIF(AVG(product_weight_g) / 1000.0, 0), 2) AS frete_medio_por_kg
  FROM item_metrics
  GROUP BY product_category_name
  HAVING COUNT(*) >= 30
)
SELECT * FROM categoria_agg ORDER BY frete_medio_por_kg DESC;


delivery_status,total_pedidos,nota_media,percentual_notas_1_2
adiantado_ou_no_prazo,89941,4.29,9.27
atraso_1_7_dias,3672,2.72,49.33
atraso_mais_de_7_dias,2863,1.7,79.30
nao_entregue,2190,1.76,77.38


### Conclusao da Pergunta 1
Preencha apos executar a consulta acima: as 3 categorias com pior `frete_medio_por_kg` e as 3
melhores; compare com `volume_medio_m3` para checar se o padrao e consistente com baixa
densidade (produtos volumosos e leves sofrem "peso cubado" das transportadoras) ou se sugere
ineficiencia de precificacao do frete. Produtos com peso/dimensao nula foram excluidos
explicitamente desta analise (ver `silver_products` no catalogo de dados).


In [ ]:
-- Pergunta 2: concentracao de atrasos e notas baixas por vendedor.
WITH pedidos_vendedor AS (
  SELECT
    f.seller_id,
    f.order_id,
    f.delivery_status,
    MAX(CAST(f.review_score AS INT)) AS review_score
  FROM ${catalog}.olist_gold.fato_vendas f
  GROUP BY f.seller_id, f.order_id, f.delivery_status
),
seller_agg AS (
  SELECT
    seller_id,
    COUNT(*) AS total_pedidos,
    SUM(CASE WHEN delivery_status IN ('atraso_1_7_dias', 'atraso_mais_de_7_dias') THEN 1 ELSE 0 END) AS pedidos_atrasados,
    SUM(CASE WHEN review_score IN (1, 2) THEN 1 ELSE 0 END) AS pedidos_nota_baixa
  FROM pedidos_vendedor
  GROUP BY seller_id
  HAVING COUNT(*) >= 10
),
ranked AS (
  SELECT *, NTILE(10) OVER (ORDER BY pedidos_atrasados DESC, pedidos_nota_baixa DESC) AS decil_risco
  FROM seller_agg
)
SELECT
  decil_risco,
  COUNT(*) AS qtd_vendedores,
  SUM(total_pedidos) AS total_pedidos,
  SUM(pedidos_atrasados) AS total_atrasados,
  SUM(pedidos_nota_baixa) AS total_nota_baixa,
  ROUND(100.0 * SUM(pedidos_atrasados) / SUM(SUM(pedidos_atrasados)) OVER (), 2) AS pct_do_total_atrasos,
  ROUND(100.0 * SUM(pedidos_nota_baixa) / SUM(SUM(pedidos_nota_baixa)) OVER (), 2) AS pct_do_total_notas_baixas
FROM ranked
GROUP BY decil_risco
ORDER BY decil_risco;


### Conclusao da Pergunta 2
Observe `decil_risco = 1` (10% piores vendedores por atrasos/notas baixas). Se
`pct_do_total_atrasos` e `pct_do_total_notas_baixas` estiverem bem acima de 10%, confirma
concentracao (padrao 80/20) em vez de um problema distribuido por toda a base. Registre: o
percentual do decil 1, o numero absoluto de vendedores nesse grupo (`qtd_vendedores`), e uma
recomendacao pratica — auditoria dos vendedores do decil 1 vs. revisao de transportadora/processo
geral, dependendo do quao concentrado o resultado for.


## Resposta executiva
Consolide aqui as duas conclusoes com os valores reais obtidos:

- **Frete vs. porte fisico:** categoria(s) com pior `frete_medio_por_kg`, comparacao com o
  `volume_medio_m3`, e se isso indica baixa densidade do produto ou oportunidade de revisar a
  precificacao de frete dessas categorias.
- **Concentracao por vendedor:** percentual de atrasos/notas baixas concentrado no decil 1 e
  recomendacao (auditoria pontual em vendedores criticos vs. revisao ampla de logistica).

Substitua os campos acima pelos valores exibidos nas consultas antes de entregar o relatorio.
